# PRISM Rebuttal - MLP probe

Closes the gap in Appendix F, which states:

> substituting a multilayer perceptron with one hidden layer of size 256
> changes the absolute ECE values but not the ordering of FMs within any
> single dataset, nor the qualitative phenomena reported in Section 4

That sentence asserts three testable claims. This notebook measures all three
against the corrected linear-probe results in `results_v2/indomain_all_v2.csv`:

1. **Ordering preserved.** Within each (dataset, fraction), does the MLP probe
   rank the eight FMs the same way the linear probe does? Measured by Kendall
   tau on AUROC and separately on ECE.
2. **Decoupling survives.** Is the AUROC-versus-calibration rank correlation
   still near zero at 1% labels and positive at 100%?
3. **Calibration inversion survives.** On LungHist700, does ECE still worsen
   for UNI, GigaPath and H-Optimus-0 as the label fraction rises?

**Protocol parity.** Identical stratified subsets, identical seeds, identical
ECE definition under both binning schemes, temperature fitted on the same
held-out validation split. The only change is the probe: a one-hidden-layer
MLP of width 256 instead of L2-regularised logistic regression.

Implemented in PyTorch so the 864 fits run on GPU in roughly an hour. Falls
back to CPU, where it takes considerably longer. Checkpointed per
(model, dataset).

In [1]:
import os, gc, glob, time, warnings
import numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from scipy.stats import kendalltau, spearmanr
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/PRISM'
EMB_DIR = f'{BASE}/embeddings'
OUT_DIR = f'{BASE}/results_v2'
CKPT    = f'{OUT_DIR}/mlp_parts'
os.makedirs(CKPT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

DATASETS = ['PCam','BRACS','CRC','MHIST','LungHist700','SPIDER-Breast']
DKEYS    = ['pcam','bracs','crc','mhist','lunghist700','spider_breast']
D2K      = dict(zip(DATASETS, DKEYS))

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS     = [42, 123, 456]
N_BINS    = 15

# MLP configuration. Width 256 is fixed by the claim in Appendix F; the
# remaining choices are stated here because the paper does not specify them.
HIDDEN       = 256
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 200
PATIENCE     = 15
BATCH        = 512

# smallest datasets first so early results arrive quickly
ORDER = ['LungHist700', 'MHIST', 'BRACS', 'CRC', 'SPIDER-Breast', 'PCam']

print(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')
print('checkpoints ->', CKPT)

Mounted at /content/drive
NVIDIA A100-SXM4-80GB
checkpoints -> /content/drive/MyDrive/PRISM/results_v2/mlp_parts


## 1. Metrics, identical to the linear-probe run

In [2]:
def _ece_edges(conf, correct, edges):
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)

def conf_correct(proba, y):
    if proba.shape[1] == 2:
        return proba[:, 1], (y == 1).astype(float)
    return proba.max(1), (proba.argmax(1) == y).astype(float)

def ece_fixed(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    return _ece_edges(c, k, np.linspace(0, 1, n_bins + 1))

def ece_adaptive(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    e = np.quantile(c, np.linspace(0, 1, n_bins + 1))
    e[0], e[-1] = 0.0, 1.0 + 1e-9
    e = np.unique(e)
    return ece_fixed(proba, y, n_bins) if len(e) < 3 else _ece_edges(c, k, e)

def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(val_logits, val_y, bounds=(0.1, 10.0)):
    idx = np.arange(len(val_y))
    def nll(T):
        p = softmax_np(val_logits / T)
        return float(-np.log(p[idx, val_y] + 1e-12).mean())
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)

def stratified_sample(labels, fraction, seed):
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked, forced = [], []
    for c in np.unique(labels):
        ci = idx_all[labels == c]
        exact = len(ci) * fraction
        n = max(1, int(exact))
        if exact < 1:
            forced.append(int(c))
        picked.extend(np.random.choice(ci, size=n, replace=False))
    return np.array(sorted(picked)), forced

def degeneracy(pred, k):
    cnt = np.bincount(pred, minlength=k)
    return float(cnt.max() / cnt.sum())

def load_emb(mkey, dkey, split):
    p = f'{EMB_DIR}/{mkey}/{dkey}'
    return (np.load(f'{p}/{split}_features.npy', mmap_mode='r'),
            np.load(f'{p}/{split}_labels.npy').astype(int))

print('metrics ready')

metrics ready


## 2. The MLP probe

One hidden layer of width 256 with ReLU, trained with Adam and early stopping
on validation loss. Where no validation split exists the last training epoch
is used and the temperature is reported as `NaN`.

In [3]:
class MLPProbe(nn.Module):
    def __init__(self, d_in, n_classes, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def logits_of(model, X, batch=8192):
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        xb = torch.as_tensor(np.asarray(X[i:i+batch], dtype=np.float32),
                             device=DEVICE)
        out.append(model(xb).float().cpu().numpy())
    return np.vstack(out)


def train_mlp(Xtr, ytr, Xva, yva, n_classes, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    Xt = torch.as_tensor(np.asarray(Xtr, dtype=np.float32), device=DEVICE)
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)

    model = MLPProbe(Xt.shape[1], n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR,
                           weight_decay=WEIGHT_DECAY)
    lossf = nn.CrossEntropyLoss()

    has_val = Xva is not None and len(yva) > 0
    if has_val:
        Xv = torch.as_tensor(np.asarray(Xva, dtype=np.float32), device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)

    n  = len(yt)
    bs = min(BATCH, n)
    best, best_state, bad = float('inf'), None, 0

    for _ in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            j = perm[i:i+bs]
            loss = lossf(model(Xt[j]), yt[j])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        if has_val:
            model.eval()
            with torch.no_grad():
                vl = float(lossf(model(Xv), yv))
            if vl < best - 1e-5:
                best, bad = vl, 0
                best_state = {k: v.detach().clone()
                              for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    del Xt, yt
    if has_val:
        del Xv, yv
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return model


print('probe ready')

probe ready


## 3. Run all 864 cells

In [4]:
def run_cell(model_name, dataset):
    mk, dk = M2K[model_name], D2K[dataset]
    Xtr, ytr = load_emb(mk, dk, 'train')
    Xte, yte = load_emb(mk, dk, 'test')
    try:
        Xva, yva = load_emb(mk, dk, 'val')
    except FileNotFoundError:
        Xva, yva = None, None

    n_classes = len(np.unique(ytr))
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, forced = stratified_sample(ytr, frac, seed)
            net = train_mlp(np.asarray(Xtr[idx]), ytr[idx],
                            Xva, yva, n_classes, seed)

            Lte   = logits_of(net, Xte)
            proba = softmax_np(Lte)
            pred  = proba.argmax(1)

            try:
                auroc = (roc_auc_score(yte, proba[:, 1]) if n_classes == 2
                         else roc_auc_score(yte, proba, multi_class='ovr',
                                            average='macro'))
            except Exception:
                auroc = np.nan

            if Xva is not None:
                T  = fit_temperature(logits_of(net, Xva), yva)
                sp = softmax_np(Lte / T)
                ece_s_fix, ece_s_ada = ece_fixed(sp, yte), ece_adaptive(sp, yte)
            else:
                T = ece_s_fix = ece_s_ada = np.nan

            rows.append(dict(
                probe='mlp', model=model_name, dataset=dataset,
                fraction=frac, seed=seed, n_train=len(idx),
                n_classes=n_classes, hidden=HIDDEN, auroc=auroc,
                f1_macro=f1_score(yte, pred, average='macro', zero_division=0),
                brier=(brier_score_loss(yte, proba[:, 1])
                       if n_classes == 2 else np.nan),
                ece_fixed=ece_fixed(proba, yte),
                ece_adaptive=ece_adaptive(proba, yte),
                temperature=T,
                ece_scaled_fixed=ece_s_fix,
                ece_scaled_adaptive=ece_s_ada,
                degeneracy_share=degeneracy(pred, n_classes),
                forced_classes=len(forced)))

            del net, Lte, proba
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    del Xtr, Xte, Xva
    gc.collect()
    df = pd.DataFrame(rows)
    df['degenerate'] = df['degeneracy_share'] > 0.99
    return df


t0 = time.time()
for dataset in ORDER:
    for model_name in MODELS:
        out = f'{CKPT}/{M2K[model_name]}__{D2K[dataset]}.csv'
        if os.path.exists(out):
            print(f'  skip: {model_name} x {dataset}')
            continue
        try:
            df = run_cell(model_name, dataset)
            df.to_csv(out, index=False)
            s = df.groupby('fraction')['auroc'].mean()
            flag = ' [DEGEN]' if df['degenerate'].any() else ''
            print(f'{model_name:>12} x {dataset:<14} '
                  f'1%={s.loc[0.01]:.4f}  100%={s.loc[1.00]:.4f}{flag}  '
                  f'({time.time()-t0:.0f}s toplam)')
        except Exception as e:
            print(f'{model_name:>12} x {dataset:<14} FAILED: '
                  f'{type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT}/*.csv'))
df_mlp = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_mlp.to_csv(f'{OUT_DIR}/mlp_probe.csv', index=False)
print(f'\n{len(parts)}/48 cells, {len(df_mlp)} runs -> mlp_probe.csv')
print('If the session dies, re-run this cell; finished cells are skipped.')

        CLIP x LungHist700    1%=0.5982  100%=0.9311  (23s toplam)
        PLIP x LungHist700    1%=0.7655  100%=0.9722  (38s toplam)
       CONCH x LungHist700    1%=0.8783  100%=0.9855  (53s toplam)
    VIRCHOW2 x LungHist700    1%=0.8383  100%=0.9878  (67s toplam)
         UNI x LungHist700    1%=0.7000  100%=0.9834  (82s toplam)
    GigaPath x LungHist700    1%=0.6797  100%=0.9844  (98s toplam)
 H-Optimus-0 x LungHist700    1%=0.6435  100%=0.9896  (113s toplam)
    MIDNIGHT x LungHist700    1%=0.7050  100%=0.9840  (126s toplam)
        CLIP x MHIST          1%=0.6388  100%=0.8724 [DEGEN]  (143s toplam)
        PLIP x MHIST          1%=0.6992  100%=0.8736 [DEGEN]  (159s toplam)
       CONCH x MHIST          1%=0.5796  100%=0.8750 [DEGEN]  (178s toplam)
    VIRCHOW2 x MHIST          1%=0.6556  100%=0.9299 [DEGEN]  (193s toplam)
         UNI x MHIST          1%=0.6653  100%=0.9136  (208s toplam)
    GigaPath x MHIST          1%=0.6691  100%=0.9049  (223s toplam)
 H-Optimus-0 x MHIST  

## 4. Claim 1: is the FM ordering preserved?

Appendix F asserts the MLP "does not change the ordering of FMs within any
single dataset". Kendall tau of 1.0 means identical ordering.

In [5]:
lin = pd.read_csv(f'{OUT_DIR}/indomain_all_v2.csv')
lin['probe'] = 'linear'
mlp = df_mlp.copy()

def ranks(df, dataset, fraction, col):
    s = (df[(df.dataset == dataset) & (df.fraction == fraction)]
         .groupby('model')[col].mean().dropna())
    return s

rows = []
for ds in DATASETS:
    for f in FRACTIONS:
        for col, asc in [('auroc', False), ('ece_scaled_fixed', True)]:
            a, b = ranks(lin, ds, f, col), ranks(mlp, ds, f, col)
            k = a.index.intersection(b.index)
            if len(k) < 3:
                continue
            tau = kendalltau(a[k].rank(ascending=asc),
                             b[k].rank(ascending=asc)).correlation
            rows.append(dict(dataset=ds, fraction=f, metric=col,
                             kendall_tau=tau, n_models=len(k),
                             identical=bool(np.isclose(tau, 1.0))))

df_rank = pd.DataFrame(rows)
df_rank.to_csv(f'{OUT_DIR}/mlp_vs_linear_ranking.csv', index=False)

for col in ['auroc', 'ece_scaled_fixed']:
    t = df_rank[df_rank.metric == col].pivot_table(
        index='dataset', columns='fraction', values='kendall_tau')
    print(f'\n=== Kendall tau, linear vs MLP ordering, {col} ===')
    print(t.round(3).to_string())
    sub = df_rank[df_rank.metric == col]
    print(f'  mean tau        : {sub.kendall_tau.mean():.3f}')
    print(f'  identical cells : {sub.identical.sum()}/{len(sub)}')

print('\nThe Appendix F claim holds only if ordering is preserved. '
      'Report what the numbers show, not what the sentence asserts.')


=== Kendall tau, linear vs MLP ordering, auroc ===
fraction        0.01   0.05   0.10   0.25   0.50   1.00
dataset                                                
BRACS          0.857  0.929  0.857  0.929  0.857  0.714
CRC            1.000  0.786  0.786  0.929  1.000  0.929
LungHist700    1.000  0.786  0.643  0.714  0.714  0.714
MHIST          0.714  0.714  0.786  0.786  0.857  0.643
PCam           0.929  0.857  0.929  0.929  0.929  0.929
SPIDER-Breast  0.929  0.786  0.786  0.929  0.929  0.929
  mean tau        : 0.845
  identical cells : 3/36

=== Kendall tau, linear vs MLP ordering, ece_scaled_fixed ===
fraction        0.01   0.05   0.10   0.25   0.50   1.00
dataset                                                
BRACS          0.286 -0.214 -0.643  0.143 -0.071  0.500
CRC            0.714  0.429  0.786  0.429  0.929  0.643
LungHist700    0.143  0.143  0.000  0.786 -0.214  0.500
MHIST          0.000 -0.071  0.000 -0.500  0.214  0.643
PCam           0.214  0.214  0.357  0.357  0.286  

## 5. Claim 2: does the decoupling survive?

In [6]:
def pooled_rho(df, fraction, ece_col='ece_scaled_fixed'):
    out = []
    for ds in DATASETS:
        s = (df[(df.dataset == ds) & (df.fraction == fraction)]
             .groupby('model')[['auroc', ece_col]].mean().dropna())
        if len(s) < 3:
            continue
        ra = s['auroc'].rank(ascending=False)
        rc = s[ece_col].rank(ascending=True)
        for m in s.index:
            out.append((ra[m], rc[m]))
    if len(out) < 4:
        return np.nan
    a, c = zip(*out)
    return spearmanr(a, c).correlation

print('Pooled within-dataset rank correlation, AUROC vs calibration')
print(f"{'frac':>6} {'linear':>9} {'MLP':>9}")
for f in FRACTIONS:
    print(f'{f:>6.2f} {pooled_rho(lin, f):>9.3f} {pooled_rho(mlp, f):>9.3f}')

print('\nDecoupling survives if the MLP column shows the same pattern: '
      'near zero or negative at 1%, positive at 100%.')

Pooled within-dataset rank correlation, AUROC vs calibration
  frac    linear       MLP
  0.01    -0.238     0.071
  0.05     0.389     0.202
  0.10     0.409     0.329
  0.25     0.444     0.349
  0.50     0.476     0.123
  1.00     0.516     0.417

Decoupling survives if the MLP column shows the same pattern: near zero or negative at 1%, positive at 100%.


## 6. Claim 3: does the LungHist700 calibration inversion survive?

In [7]:
print('LungHist700, raw ECE across label fraction\n')
for m in ['UNI', 'GigaPath', 'H-Optimus-0', 'VIRCHOW2', 'CONCH', 'MIDNIGHT',
          'PLIP', 'CLIP']:
    line = f'  {m:>12}'
    for probe, df in [('linear', lin), ('mlp', mlp)]:
        v = (df[(df.model == m) & (df.dataset == 'LungHist700')]
             .groupby('fraction')['ece_fixed'].mean().sort_index())
        if len(v) < 2:
            line += f'  {probe}: n/a'
            continue
        trend = 'WORSENS' if v.iloc[-1] > v.iloc[0] else 'improves'
        line += f'  |  {probe}: {v.iloc[0]:.3f} -> {v.iloc[-1]:.3f} {trend}'
    print(line)

print('\n\n=== ECE level shift, linear vs MLP, averaged over cells ===')
key = ['model','dataset','fraction']
j = (lin.groupby(key)['ece_fixed'].mean().rename('linear')
     .to_frame()
     .join(mlp.groupby(key)['ece_fixed'].mean().rename('mlp'), how='inner'))
j['delta'] = j['mlp'] - j['linear']
print(j.groupby(level='fraction')['delta']
       .agg(['mean','std','min','max']).round(4).to_string())

print('\n=== AUROC level shift ===')
ja = (lin.groupby(key)['auroc'].mean().rename('linear').to_frame()
      .join(mlp.groupby(key)['auroc'].mean().rename('mlp'), how='inner'))
ja['delta'] = ja['mlp'] - ja['linear']
print(ja.groupby(level='fraction')['delta']
        .agg(['mean','std','min','max']).round(4).to_string())

both = pd.concat([
    lin[['probe','model','dataset','fraction','seed','auroc','f1_macro',
         'ece_fixed','ece_scaled_fixed','temperature','degeneracy_share']],
    mlp[['probe','model','dataset','fraction','seed','auroc','f1_macro',
         'ece_fixed','ece_scaled_fixed','temperature','degeneracy_share']],
], ignore_index=True)
both.to_csv(f'{OUT_DIR}/mlp_vs_linear.csv', index=False)
print('\nSaved -> mlp_vs_linear.csv, mlp_vs_linear_ranking.csv, mlp_probe.csv')

LungHist700, raw ECE across label fraction

           UNI  |  linear: 0.213 -> 0.313 WORSENS  |  mlp: 0.144 -> 0.086 improves
      GigaPath  |  linear: 0.186 -> 0.312 WORSENS  |  mlp: 0.191 -> 0.067 improves
   H-Optimus-0  |  linear: 0.161 -> 0.314 WORSENS  |  mlp: 0.114 -> 0.096 improves
      VIRCHOW2  |  linear: 0.294 -> 0.264 improves  |  mlp: 0.124 -> 0.067 improves
         CONCH  |  linear: 0.331 -> 0.208 improves  |  mlp: 0.115 -> 0.070 improves
      MIDNIGHT  |  linear: 0.173 -> 0.195 WORSENS  |  mlp: 0.123 -> 0.089 improves
          PLIP  |  linear: 0.307 -> 0.224 improves  |  mlp: 0.078 -> 0.107 WORSENS
          CLIP  |  linear: 0.067 -> 0.109 WORSENS  |  mlp: 0.066 -> 0.112 WORSENS


=== ECE level shift, linear vs MLP, averaged over cells ===
            mean     std     min     max
fraction                                
0.01     -0.1275  0.1284 -0.4286  0.0282
0.05     -0.0765  0.0725 -0.2398  0.0654
0.10     -0.0715  0.0565 -0.2017  0.0595
0.25     -0.0574  0.0611

## 7. What to report

Three outcomes, all reportable.

**All three claims hold.** Appendix F was correct and we now have the table
that supports it. Add it to the supplementary materials and cite it in the
rebuttal.

**Ordering shifts but the phenomena survive.** Weaken the sentence to what the
data supports: absolute values move, the qualitative findings do not. State
the Kendall tau values.

**A phenomenon does not survive.** Say so plainly and bound the claim to the
linear-probe protocol. Reviewer tp5b reads code, and an overstated appendix is
more costly than a bounded one.